# Per-Cohort MICE Imputation — Numeric & Categorical

Fills missing values in `Final_metabric_data.tsv` using MICE, applied **independently within each cohort** so that no cohort's data informs another's imputation.



In [1]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OrdinalEncoder

RANDOM_STATE = 42
MICE_MAX_ITER = 10

NUMERIC_IMPUTE = [
    'Tumor Size',
]

CATEGORICAL_IMPUTE = [
    'Type of Breast Surgery',
    'Primary Tumor Laterality',
    'Pam50 + Claudin-low subtype',
    'Cellularity',
]

ALL_IMPUTE_COLS = NUMERIC_IMPUTE + CATEGORICAL_IMPUTE

## 1. Load data

In [2]:
df = pd.read_csv('../data/Final_metabric_data.tsv', sep='\t')
print(f'Shape: {df.shape}')
print('\nMissing values per column (only columns with any missing):')
missing = df.isna().sum()
print(missing[missing > 0].to_string())
print(f'\nCohorts: {sorted(df["Cohort"].dropna().unique())}')

Shape: (1981, 30)

Missing values per column (only columns with any missing):
Tumor Size                       26
Radio Therapy                     1
Inferred Menopausal State         1
Integrative Cluster               1
Hormone Therapy                   1
HER2 Status                       1
HER2 status measured by SNP6      1
Pam50 + Claudin-low subtype       1
Chemotherapy                      1
Type of Breast Surgery           26
Cellularity                      64
Primary Tumor Laterality        111
Relapse Free Status               1
Patient's Vital Status            1

Cohorts: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]


## 2. Fit OrdinalEncoder on the full dataset

We fit on all cohorts combined so the integer codes are consistent across cohorts even though imputation runs per cohort.

In [3]:
ordinal = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=np.nan,
    encoded_missing_value=np.nan,
)
ordinal.fit(df[CATEGORICAL_IMPUTE])

print('Category mappings:')
for col, cats in zip(CATEGORICAL_IMPUTE, ordinal.categories_):
    print(f'  {col}: {list(cats)}')

Category mappings:
  Type of Breast Surgery: ['BREAST CONSERVING', 'MASTECTOMY', nan]
  Primary Tumor Laterality: ['Left', 'Right', nan]
  Pam50 + Claudin-low subtype: ['Basal', 'Her2', 'LumA', 'LumB', 'NC', 'Normal', 'claudin-low', nan]
  Cellularity: ['High', 'Low', 'Moderate', nan]


## 3. Per-cohort MICE imputation

For each cohort we:
1. Build a matrix of `[numeric | ordinal-encoded categorical]` (NaN preserved).
2. Fit + transform MICE on that cohort alone.
3. Round ordinal columns to the nearest valid integer index.
4. Inverse-transform ordinal columns back to original category strings.
5. Write results back into the dataframe.

In [4]:
df_out = df.copy()

n_num = len(NUMERIC_IMPUTE)
category_sizes = [len(cats) for cats in ordinal.categories_]

cohorts = sorted(df['Cohort'].dropna().unique())
print(f'Imputing cohorts: {cohorts}\n')

for cohort in cohorts:
    mask = df['Cohort'] == cohort
    idx  = df.index[mask]
    sub  = df.loc[mask]

    # --- build imputation matrix ---
    num_block = sub[NUMERIC_IMPUTE].to_numpy(dtype=float)
    cat_block = ordinal.transform(sub[CATEGORICAL_IMPUTE])
    full_mat  = np.hstack([num_block, cat_block])

    pre_missing = np.isnan(full_mat).sum()

    # --- MICE ---
    imputer = IterativeImputer(random_state=RANDOM_STATE, max_iter=MICE_MAX_ITER)
    imputed = imputer.fit_transform(full_mat)

    post_missing = np.isnan(imputed).sum()
    print(f'Cohort {cohort:3.0f} | rows={len(sub):4d} | '
          f'missing before={pre_missing:3d} | missing after={post_missing}')

    # --- write back numeric ---
    for j, col in enumerate(NUMERIC_IMPUTE):
        df_out.loc[idx, col] = imputed[:, j]

    # --- round, clip, inverse-transform categorical ---
    cat_imputed = imputed[:, n_num:]
    cat_rounded = np.round(cat_imputed).astype(int)
    for i, size in enumerate(category_sizes):
        cat_rounded[:, i] = np.clip(cat_rounded[:, i], 0, size - 1)

    cat_labels = ordinal.inverse_transform(cat_rounded.astype(float))
    for j, col in enumerate(CATEGORICAL_IMPUTE):
        df_out.loc[idx, col] = cat_labels[:, j]

print('\nDone.')

Imputing cohorts: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]

Cohort   1 | rows= 522 | missing before= 80 | missing after=0
Cohort   2 | rows= 288 | missing before= 30 | missing after=0
Cohort   3 | rows= 763 | missing before= 23 | missing after=0
Cohort   4 | rows= 238 | missing before= 84 | missing after=0
Cohort   5 | rows= 170 | missing before= 11 | missing after=0

Done.


## 4. Verify — no missing values remain in imputed columns

In [10]:
print('Missing values in imputed columns after per-cohort MICE:')
print(df_out[ALL_IMPUTE_COLS].isna().sum().to_string())


print('\nValue counts per imputed categorical column:')
for col in CATEGORICAL_IMPUTE:
    print(f'\n  {col}:')
    print(df_out[col].value_counts().to_string())

Missing values in imputed columns after per-cohort MICE:
Tumor Size                     0
Type of Breast Surgery         0
Primary Tumor Laterality       0
Pam50 + Claudin-low subtype    0
Cellularity                    0

Value counts per imputed categorical column:

  Type of Breast Surgery:
Type of Breast Surgery
MASTECTOMY           1192
BREAST CONSERVING     789

  Primary Tumor Laterality:
Primary Tumor Laterality
Left     1069
Right     912

  Pam50 + Claudin-low subtype:
Pam50 + Claudin-low subtype
LumA           700
LumB           476
Her2           224
claudin-low    218
Basal          209
Normal         148
NC               6

  Cellularity:
Cellularity
High        966
Moderate    737
Low         278


## 5. Quick sanity check — original vs imputed rows

In [6]:
was_missing_any = df[ALL_IMPUTE_COLS].isna().any(axis=1)
changed_rows = df_out[was_missing_any][['Cohort'] + ALL_IMPUTE_COLS]
original_rows = df[was_missing_any][['Cohort'] + ALL_IMPUTE_COLS]

print(f'Rows that had at least one missing value: {was_missing_any.sum()}')
print('\nOriginal (first 10 rows with any missing):')
print(original_rows.head(10).to_string())
print('\nImputed (same rows):')
print(changed_rows.head(10).to_string())

Rows that had at least one missing value: 199

Original (first 10 rows with any missing):
     Cohort  Tumor Size Type of Breast Surgery Primary Tumor Laterality Pam50 + Claudin-low subtype Cellularity
0       1.0        22.0             MASTECTOMY                    Right                 claudin-low         NaN
26      1.0        50.0             MASTECTOMY                      NaN                        LumB        High
40      1.0        45.0                    NaN                      NaN                       Basal        High
57      1.0        13.0      BREAST CONSERVING                    Right                      Normal         NaN
87      1.0        23.0      BREAST CONSERVING                    Right                      Normal         NaN
89      1.0        27.0      BREAST CONSERVING                      NaN                       Basal        High
101     1.0        20.0             MASTECTOMY                    Right                        LumA         NaN
129     1.0   

## 6. Save imputed dataset

In [7]:
out_path = '../data/Final_metabric_data_mice_imputed.tsv'
df_out.to_csv(out_path, sep='\t', index=False)
print(f'Saved → {out_path}')
print(f'Shape: {df_out.shape}')

Saved → ../data/Final_metabric_data_mice_imputed.tsv
Shape: (1981, 30)
